# NESTA Tutorial with Toy Example

This notebook runs the NESTA analysis on a compact, simulation-derived dataset.
The same TWAS evidence is combined with two different cell type-specific co-expression networks and mean-expression profiles.

The goal is to show how cellular network context changes gene prioritization. `Final.Heat` is the primary prioritization score; `delta_NESTA` is an auxiliary score for interpreting network-driven changes.

## 1. Set paths

Run this notebook from the repository root or from the `tutorial/` directory. Generated files are written under `tutorial/output_notebook/`, which is ignored by Git.

In [ ]:
if (basename(getwd()) == "tutorial") setwd("..")
stopifnot(file.exists(file.path("Analysis", "Nesta.R")))

repo_root <- normalizePath(getwd())
tutorial_dir <- file.path(repo_root, "tutorial")
data_dir <- file.path(tutorial_dir, "data")
output_dir <- file.path(tutorial_dir, "output_notebook")
dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)

c(repo_root = repo_root, output_dir = output_dir)

## 2. Load the small input tables

The toy example contains 18 genes. The TWAS evidence is shared across both analyses; only the cell type-specific network and expression context change.

In [ ]:
suppressPackageStartupMessages(library(data.table))

twas <- fread(file.path(data_dir, "simulated_twas_results.tsv"))
twas

In [ ]:
network_A <- fread(file.path(data_dir, "cell_type_A_network.tsv"))
network_B <- fread(file.path(data_dir, "cell_type_B_network.tsv"))
expression_A <- fread(file.path(data_dir, "cell_type_A_mean_expression.tsv"))
expression_B <- fread(file.path(data_dir, "cell_type_B_mean_expression.tsv"))

data.table(
  cell_type = c("A", "B"),
  genes = c(length(unique(c(network_A$from, network_A$to))),
            length(unique(c(network_B$from, network_B$to)))),
  edges = c(nrow(network_A), nrow(network_B)),
  mean_expression_range = c(
    sprintf("%.1f–%.1f", min(expression_A$Mean_expression), max(expression_A$Mean_expression)),
    sprintf("%.1f–%.1f", min(expression_B$Mean_expression), max(expression_B$Mean_expression))
  )
)

## 3. Inspect how the cellular contexts differ

The two mean-expression profiles emphasize different simulated gene modules. This expression context is combined with the same signed TWAS Z-scores during initialization.

In [ ]:
expression_comparison <- merge(expression_A, expression_B, by = "SYMBOL",
                               suffixes = c("_A", "_B"))
expression_comparison[, expression_difference := Mean_expression_A - Mean_expression_B]
expression_comparison[order(-abs(expression_difference))]

## 4. Define a small wrapper around the public NESTA CLI

This function calls `Analysis/Nesta.R`; the notebook does not reimplement the NESTA. The edge list and mean-expression table provide a lightweight cell type-specific workflow without requiring a large Seurat object.

In [ ]:
run_nesta <- function(cell_type) {
  prefix <- paste0("cell_type_", cell_type)
  args <- c(
    file.path(repo_root, "Analysis", "Nesta.R"),
    "--TWAS_res", file.path(data_dir, "simulated_twas_results.tsv"),
    "--Reference_net", file.path(data_dir, paste0(prefix, "_network.tsv")),
    "--Is_expression_network", "NO",
    "--Initial_weight_mode", "nesta_expression_weighted",
    "--Mean_expression", file.path(data_dir, paste0(prefix, "_mean_expression.tsv")),
    "--Diffuse_method", "raw",
    "--Diffuse_nperm", "50",
    "--check_bias", "FALSE",
    "--edge_cutoff", "0",
    "--Analysis_name", paste0("Cell_type_", cell_type),
    "--out_dir", output_dir,
    "--prefix", prefix
  )
  status <- system2("Rscript", args = args)
  if (!identical(status, 0L)) stop("NESTA failed for Cell type ", cell_type)
  invisible(file.path(output_dir, paste0(prefix, "_scores.tsv")))
}

## 5. Run NESTA independently for each cell type

In [ ]:
scores_A_path <- run_nesta("A")
scores_B_path <- run_nesta("B")

## 6. Read and rank the outputs

Genes are ranked here by the absolute magnitude of `Final.Heat`, while retaining its directions.

In [ ]:
scores_A <- fread(scores_A_path)
scores_B <- fread(scores_B_path)
scores <- rbindlist(list(scores_A, scores_B), fill = TRUE)
scores[, abs_Final_Heat := abs(Final.Heat)]
setorder(scores, Analysis_name, -abs_Final_Heat, SYMBOL)
scores[, rank_by_Final_Heat := seq_len(.N), by = Analysis_name]

scores[rank_by_Final_Heat <= 5,
       .(Analysis_name, rank_by_Final_Heat, SYMBOL, TWAS.Z,
         Mean_expression, Final.Heat, delta_NESTA)]

## 7. Visualize the cell type-specific rankings

The same TWAS input produces distinct top-ranked profiles because the two cellular networks and expression profiles differ.

In [ ]:
top <- scores[rank_by_Final_Heat <= 8]
old_par <- par(mfrow = c(1, 2), mar = c(5, 7, 3, 1))
for (ct in c("Cell_type_A", "Cell_type_B")) {
  x <- top[Analysis_name == ct][order(Final.Heat)]
  barplot(x$Final.Heat, names.arg = x$SYMBOL, horiz = TRUE, las = 1,
          col = ifelse(x$Final.Heat >= 0, "#D94A4A", "#2563A6"),
          border = NA, main = ct, xlab = "Final Heat")
  abline(v = 0, col = "#7A8792")
}
par(old_par)

## 8. Inspect network-driven re-prioritization

`delta_NESTA` is used for interpretation. Large absolute values identify genes whose final network-informed score differs substantially from the original TWAS Z-score.

In [ ]:
scores[, abs_delta_NESTA := abs(delta_NESTA)]
scores[order(Analysis_name, -abs_delta_NESTA),
       head(.SD, 5), by = Analysis_name,
       .SDcols = c("SYMBOL", "TWAS.Z", "Final.Heat", "delta_NESTA")]

## 9. Validate the tutorial result

The validation checks required columns, finite scores, the `delta_NESTA` definition, different cell type-specific top-five profiles, and the expected gene ordering.

In [ ]:
validation_status <- system2(
  "Rscript",
  args = c(file.path(tutorial_dir, "validate_results.R"), output_dir)
)
stopifnot(identical(validation_status, 0L))

## Interpretation

Cell type A emphasizes the first simulated module, whereas Cell type B emphasizes the second. This illustrates the central NESTA idea: gene prioritization reflects both signed genetic evidence and the structure of the relevant cell type-specific network.

For the full simulation and threshold-sensitivity workflow, see `simulation_study/README.md`.